# Clase 068 — Análisis de errores

Dejamos de mirar el accuracy global y auditamos **dónde** se equivoca el clasificador: matriz de confusión normalizada por fila, pares de clases confundidas, inspección visual de errores y análisis por slices como puerta al *data-centric AI*.

Requiere: `numpy`, `scikit-learn`, `matplotlib`.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.datasets import load_digits
from sklearn.linear_model import SGDClassifier
from sklearn.model_selection import train_test_split, cross_val_predict
from sklearn.metrics import confusion_matrix, accuracy_score

np.random.seed(42)

digits = load_digits()
X, y = digits.data, digits.target
Xtr, Xte, ytr, yte = train_test_split(X, y, test_size=0.3, stratify=y, random_state=42)
print('train', Xtr.shape, '| test', Xte.shape)

## 1. Matriz de confusión normalizada por fila

Predecimos out-of-fold sobre el train con `cross_val_predict`. Normalizamos por fila (`C[i,j] = P(pred=j | real=i)`) y ponemos ceros en la diagonal para que los **errores** salten a la vista.

In [ ]:
sgd = SGDClassifier(random_state=42)
y_pred = cross_val_predict(sgd, Xtr, ytr, cv=3, n_jobs=1)

cm = confusion_matrix(ytr, y_pred)
cm_norm = cm / cm.sum(axis=1, keepdims=True)
assert np.allclose(cm_norm.sum(axis=1), 1.0)   # cada fila suma 1

cm_err = cm_norm.copy()
np.fill_diagonal(cm_err, 0)

fig, ax = plt.subplots(figsize=(6, 5))
im = ax.matshow(cm_err, cmap='Reds')
fig.colorbar(im, ax=ax, fraction=0.046)
ax.set_xlabel('predicho')
ax.set_ylabel('real')
ax.set_title('Errores (matriz normalizada por fila, diagonal=0)', pad=20)
plt.tight_layout()
plt.show()

## 2. Top-5 confusiones

Ordenamos los off-diagonals normalizados de mayor a menor: es la lista priorizada de qué pares atacar primero.

In [ ]:
pares = []
for i in range(10):
    for j in range(10):
        if i != j:
            pares.append((cm_err[i, j], i, j))
pares.sort(reverse=True)

print('Top-5 confusiones:')
for val, i, j in pares[:5]:
    print(f'  real={i} -> pred={j}: {val*100:.1f}%')

peor_val, peor_real, peor_pred = pares[0]
print(f'\npar peor: real={peor_real} confundido con pred={peor_pred}')

## 3. Galería de errores del par peor

Mostramos ejemplos reales donde el modelo confundió el par dominante. Inspeccionarlos a mano revela si es ruido de label, ambigüedad o falta de capacidad.

In [ ]:
mask = (ytr == peor_real) & (y_pred == peor_pred)
idxs = np.where(mask)[0]
print(f'ejemplos real={peor_real} pred={peor_pred}: {len(idxs)}')

n = min(9, len(idxs))
fig, axes = plt.subplots(3, 3, figsize=(5, 5))
for k, ax in enumerate(axes.ravel()):
    if k < n:
        ax.imshow(Xtr[idxs[k]].reshape(8, 8), cmap='binary')
        ax.set_title(f'real={peor_real} pred={peor_pred}', fontsize=8)
    ax.axis('off')
plt.suptitle('Galeria de errores (hard examples)')
plt.tight_layout()
plt.show()

assert n >= 1

## 4. Slice por intensidad de trazo

La suma de píxeles aproxima el "grosor" del dígito. Dividimos el train en terciles y reportamos accuracy por tercil: el promedio global puede esconder subgrupos difíciles.

In [ ]:
intensidad = Xtr.sum(axis=1)
q1, q2 = np.quantile(intensidad, [1/3, 2/3])
terciles = np.digitize(intensidad, [q1, q2])   # 0,1,2

print('accuracy por tercil de intensidad:')
for t, nombre in enumerate(['fino', 'medio', 'grueso']):
    m = terciles == t
    acc = accuracy_score(ytr[m], y_pred[m])
    print(f'  {nombre:7s}: acc={acc:.3f}  (n={m.sum()})')

acc_global = accuracy_score(ytr, y_pred)
print(f'\naccuracy global: {acc_global:.3f}')

## 5. Barras de accuracy por slice

Visualizamos el accuracy por tercil contra el global. Si un tercil queda muy por debajo, hay un subgrupo que el modelo maneja peor.

In [ ]:
accs = [accuracy_score(ytr[terciles == t], y_pred[terciles == t]) for t in range(3)]
fig, ax = plt.subplots(figsize=(6, 4))
ax.bar(['fino', 'medio', 'grueso'], accs, color=['#37a', '#3a7', '#a73'])
ax.axhline(acc_global, color='black', ls='--', label=f'global={acc_global:.3f}')
ax.set_ylabel('accuracy')
ax.set_ylim(0, 1.05)
ax.set_title('Accuracy por slice de intensidad')
ax.legend()
plt.tight_layout()
plt.show()

## Ejercicios

1. **Top-3 justificado.** Extraé los 3 pares más confundidos y explicá numéricamente por qué encabezan la lista.
2. **Galería 16.** Mostrá una grilla 4×4 de errores del par peor con título `real=X pred=Y`. ¿Te parecen ambiguos o mal labelados?
3. **Otro slice.** Definí una variable derivada distinta (posición del centro de masa) y reportá accuracy por tercil.
4. **Hipótesis.** En un párrafo: ¿el error dominante es de **modelo** o de **datos**? ¿Qué probarías en la próxima iteración?

## Conclusiones

- Normalizá la matriz **por fila** (`axis=1`): la diagonal es el recall por clase, los off-diagonals la distribución de errores.
- Poné **ceros en la diagonal** antes de plotear para que los errores no queden tapados.
- Los errores **no son uniformes**: hay 2-3 pares dominantes que priorizar (top-k confusiones).
- El **slice analysis** revela subgrupos que el accuracy global entierra.
- Hacé el análisis siempre sobre datos **out-of-fold** (`cross_val_predict`), nunca sobre train memorizado; muchas veces la próxima mejora está en los **datos**, no en el modelo.